# 🤖 A2A Agent Card Generator

**Official A2A Protocol Implementation**  
*Created by Paola Di Maio*

Interactive Google Colab notebook for generating A2A-compliant Agent Cards through user-friendly forms.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a2aproject/a2a-samples/blob/main/tools/generators/A2A_Standard_Agent_Card_Generator.ipynb)

See `README.md` for full documentation.

---

In [ ]:
# 📦 Setup - Run this first!
!pip install ipywidgets -q

In [ ]:
import json

import ipywidgets as widgets
from google.colab import files
from IPython.display import HTML, clear_output, display

print("✅ Setup complete! Ready to generate A2A Agent Cards.")

In [ ]:
# 🎨 Global Storage for Agent Card Data
agent_card_data = {
    "protocolVersion": "1.0",
    "name": "",
    "description": "",
    "version": "1.0.0",
    "provider": {"organization": "", "url": ""},
    "supportedInterfaces": [],
    "capabilities": {
        "streaming": False,
        "pushNotifications": False,
        "stateTransitionHistory": False,
    },
    "defaultInputModes": [],
    "defaultOutputModes": [],
    "skills": [],
    "documentationUrl": "",
    "iconUrl": "",
    "supportsExtendedAgentCard": False,
}

# Common media types
MEDIA_TYPES = [
    "text/plain",
    "application/json",
    "image/jpeg",
    "image/png",
    "image/webp",
    "audio/mpeg",
    "video/mp4",
    "application/pdf",
    "text/html",
    "text/markdown",
]

# Protocol bindings
PROTOCOL_BINDINGS = ["JSONRPC", "GRPC", "HTTP+JSON"]

print("✅ Data structures initialized!")

---

# 🔵 BASIC INFORMATION

**Core agent identity and metadata**

In [ ]:
# 🔵 BASIC INFORMATION

display(HTML("<h2 style='color: #3b82f6;'>🔵 Basic Information</h2>"))
display(HTML("<p style='color: #64748b;'>Core agent identity and description</p>"))

agent_name = widgets.Text(
    description="Agent Name:",
    placeholder="e.g., Recipe Assistant",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px"),
)

agent_version = widgets.Text(
    value="1.0.0",
    description="Version:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="300px"),
)

agent_description = widgets.Textarea(
    description="Description:",
    placeholder="Describe what this agent does...",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="600px", height="80px"),
)

provider_org = widgets.Text(
    description="Organization:",
    placeholder="e.g., OpenAI, Google",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="400px"),
)

provider_url = widgets.Text(
    description="Provider URL:",
    placeholder="https://example.com",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px"),
)

doc_url = widgets.Text(
    description="Documentation:",
    placeholder="https://docs.example.com (optional)",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px"),
)

icon_url = widgets.Text(
    description="Icon URL:",
    placeholder="https://example.com/icon.png (optional)",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px"),
)


def update_basic_info():
    agent_card_data["name"] = agent_name.value
    agent_card_data["version"] = agent_version.value
    agent_card_data["description"] = agent_description.value
    agent_card_data["provider"]["organization"] = provider_org.value
    agent_card_data["provider"]["url"] = provider_url.value
    agent_card_data["documentationUrl"] = doc_url.value
    agent_card_data["iconUrl"] = icon_url.value


basic_info_widgets = [
    agent_name,
    agent_version,
    agent_description,
    provider_org,
    provider_url,
    doc_url,
    icon_url,
]
for widget in basic_info_widgets:
    widget.observe(lambda change: update_basic_info(), "value")

display(agent_name, agent_version, agent_description)
display(HTML("<h3 style='margin-top: 20px;'>Provider Information</h3>"))
display(provider_org, provider_url)
display(HTML("<h3 style='margin-top: 20px;'>Optional URLs</h3>"))
display(doc_url, icon_url)

---

# 🟢 SERVICE ENDPOINTS

**Define where and how agents can connect**

In [ ]:
# 🟢 SERVICE ENDPOINTS

display(HTML("<h2 style='color: #10b981;'>🟢 Service Endpoints</h2>"))
display(HTML("<p style='color: #64748b;'>Configure interfaces and protocols</p>"))

interfaces_output = widgets.Output()
interfaces_list = []  # Store interface configurations

interface_url = widgets.Text(
    description="Endpoint URL:",
    placeholder="https://api.example.com/a2a/v1",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="500px"),
)

interface_protocol = widgets.Dropdown(
    options=PROTOCOL_BINDINGS,
    value="JSONRPC",
    description="Protocol:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="300px"),
)

interface_tenant = widgets.Text(
    description="Tenant (opt):",
    placeholder="tenant-id (optional)",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="300px"),
)

add_interface_button = widgets.Button(
    description="➕ Add Interface",
    button_style="success",
    layout=widgets.Layout(width="200px"),
)


def update_interfaces_display():
    with interfaces_output:
        clear_output()
        if interfaces_list:
            print("\n📋 Configured Interfaces:")
            for i, iface in enumerate(interfaces_list):
                tenant = iface.get("tenant")
                tenant_str = f" (tenant: {tenant})" if tenant else ""
                print(
                    f"  {i + 1}. {iface['protocolBinding']} → "
                    f"{iface['url']}{tenant_str}"
                )
            print(f"\n✅ {len(interfaces_list)} interface(s) configured")
        else:
            print("⚠️ No interfaces configured yet. Add at least one!")


def add_interface(b):
    if not interface_url.value:
        print("⚠️ Please enter an endpoint URL")
        return

    new_interface = {
        "url": interface_url.value,
        "protocolBinding": interface_protocol.value,
    }
    if interface_tenant.value:
        new_interface["tenant"] = interface_tenant.value

    interfaces_list.append(new_interface)
    agent_card_data["supportedInterfaces"] = interfaces_list

    interface_url.value = ""
    interface_tenant.value = ""

    update_interfaces_display()


add_interface_button.on_click(add_interface)

display(interface_url)
display(widgets.HBox([interface_protocol, interface_tenant]))
display(add_interface_button)
display(interfaces_output)

update_interfaces_display()

---

# 🟡 CAPABILITIES

**Specify what features the agent supports**

In [ ]:
# 🟡 CAPABILITIES

display(HTML("<h2 style='color: #f59e0b;'>🟡 Capabilities</h2>"))
display(HTML("<p style='color: #64748b;'>Configure optional A2A features</p>"))

cap_streaming = widgets.Checkbox(
    value=False,
    description="Streaming Support",
    style={"description_width": "initial"},
)

cap_push = widgets.Checkbox(
    value=False,
    description="Push Notifications",
    style={"description_width": "initial"},
)

cap_history = widgets.Checkbox(
    value=False,
    description="State Transition History",
    style={"description_width": "initial"},
)

cap_extended = widgets.Checkbox(
    value=False,
    description="Extended Agent Card Support",
    style={"description_width": "initial"},
)


def update_capabilities():
    agent_card_data["capabilities"]["streaming"] = cap_streaming.value
    agent_card_data["capabilities"]["pushNotifications"] = cap_push.value
    agent_card_data["capabilities"]["stateTransitionHistory"] = cap_history.value
    agent_card_data["supportsExtendedAgentCard"] = cap_extended.value


for widget in [cap_streaming, cap_push, cap_history, cap_extended]:
    widget.observe(lambda change: update_capabilities(), "value")

display(cap_streaming, cap_push, cap_history, cap_extended)

---

# 🟣 INTERACTION MODES

**Define supported input/output media types**

In [ ]:
# 🟣 INTERACTION MODES

display(HTML("<h2 style='color: #a855f7;'>🟣 Interaction Modes</h2>"))
display(HTML("<p style='color: #64748b;'>Select supported media types</p>"))

input_modes = widgets.SelectMultiple(
    options=MEDIA_TYPES,
    value=["text/plain"],
    description="Input Modes:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="400px", height="150px"),
)

output_modes = widgets.SelectMultiple(
    options=MEDIA_TYPES,
    value=["text/plain"],
    description="Output Modes:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="400px", height="150px"),
)


def update_modes():
    agent_card_data["defaultInputModes"] = list(input_modes.value)
    agent_card_data["defaultOutputModes"] = list(output_modes.value)


input_modes.observe(lambda change: update_modes(), "value")
output_modes.observe(lambda change: update_modes(), "value")

display(HTML("<p><em>Hold Ctrl/Cmd to select multiple</em></p>"))
display(input_modes, output_modes)

update_modes()

---

# 🔶 SKILLS

**Define agent capabilities and functions**

In [ ]:
# 🔶 SKILLS

display(HTML("<h2 style='color: #f97316;'>🔶 Skills</h2>"))
display(HTML("<p style='color: #64748b;'>Define what your agent can do</p>"))

skills_output = widgets.Output()
skills_list = []  # Store skills

skill_id = widgets.Text(
    description="Skill ID:",
    placeholder="e.g., recipe-search",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="400px"),
)

skill_name = widgets.Text(
    description="Name:",
    placeholder="e.g., Recipe Search",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="400px"),
)

skill_description = widgets.Textarea(
    description="Description:",
    placeholder="What does this skill do?",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="500px", height="60px"),
)

skill_tags = widgets.Text(
    description="Tags:",
    placeholder="cooking, recipes, search (comma-separated)",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="500px"),
)

skill_examples = widgets.Textarea(
    description="Examples:",
    placeholder="One example per line",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="500px", height="60px"),
)

add_skill_button = widgets.Button(
    description="➕ Add Skill",
    button_style="warning",
    layout=widgets.Layout(width="200px"),
)


def update_skills_display():
    with skills_output:
        clear_output()
        if skills_list:
            print("\n🎯 Configured Skills:")
            for i, skill in enumerate(skills_list):
                print(f"  {i + 1}. {skill['name']} ({skill['id']})")
                print(f"     Tags: {', '.join(skill['tags'])}")
            print(f"\n✅ {len(skills_list)} skill(s) configured")
        else:
            print("⚠️ No skills configured yet. Add at least one!")


def add_skill(b):
    if not skill_id.value or not skill_name.value:
        print("⚠️ Please enter skill ID and name")
        return

    new_skill = {
        "id": skill_id.value,
        "name": skill_name.value,
        "description": skill_description.value,
        "tags": [tag.strip() for tag in skill_tags.value.split(",") if tag.strip()],
    }

    if skill_examples.value:
        new_skill["examples"] = [
            ex.strip() for ex in skill_examples.value.split("\n") if ex.strip()
        ]

    skills_list.append(new_skill)
    agent_card_data["skills"] = skills_list

    skill_id.value = ""
    skill_name.value = ""
    skill_description.value = ""
    skill_tags.value = ""
    skill_examples.value = ""

    update_skills_display()


add_skill_button.on_click(add_skill)

display(skill_id, skill_name, skill_description, skill_tags, skill_examples)
display(add_skill_button)
display(skills_output)

update_skills_display()

---

# 🟪 GENERATE AGENT CARD

**Create and download your A2A Agent Card JSON**

In [ ]:
# 🟪 JSON GENERATION & EXPORT


def generate_agent_card_json():
    """Generate the complete A2A Agent Card JSON."""
    update_basic_info()
    update_capabilities()
    update_modes()

    card = {
        "protocolVersion": agent_card_data["protocolVersion"],
        "name": agent_card_data["name"],
        "description": agent_card_data["description"],
        "version": agent_card_data["version"],
    }

    provider = agent_card_data["provider"]
    if provider["organization"] or provider["url"]:
        card["provider"] = agent_card_data["provider"]

    card["supportedInterfaces"] = agent_card_data["supportedInterfaces"]
    card["capabilities"] = agent_card_data["capabilities"]
    card["defaultInputModes"] = agent_card_data["defaultInputModes"]
    card["defaultOutputModes"] = agent_card_data["defaultOutputModes"]
    card["skills"] = agent_card_data["skills"]

    if agent_card_data["documentationUrl"]:
        card["documentationUrl"] = agent_card_data["documentationUrl"]
    if agent_card_data["iconUrl"]:
        card["iconUrl"] = agent_card_data["iconUrl"]
    if agent_card_data["supportsExtendedAgentCard"]:
        card["supportsExtendedAgentCard"] = True

    return card


json_output = widgets.Output()

generate_button = widgets.Button(
    description="🟪 Generate JSON",
    button_style="primary",
    layout=widgets.Layout(width="200px", height="40px"),
)

download_button = widgets.Button(
    description="💾 Download JSON",
    button_style="success",
    layout=widgets.Layout(width="200px", height="40px"),
)

current_json = {}


def validate_card():
    """Validate required fields."""
    errors = []
    if not agent_card_data["name"]:
        errors.append("❌ Agent Name is required")
    if not agent_card_data["description"]:
        errors.append("❌ Description is required")
    if not agent_card_data["supportedInterfaces"]:
        errors.append("❌ At least one interface is required")
    if not agent_card_data["defaultInputModes"]:
        errors.append("❌ At least one input mode is required")
    if not agent_card_data["defaultOutputModes"]:
        errors.append("❌ At least one output mode is required")
    if not agent_card_data["skills"]:
        errors.append("❌ At least one skill is required")
    return errors


def on_generate_click(b):
    global current_json
    with json_output:
        clear_output()
        errors = validate_card()
        if errors:
            print("⚠️ Validation Errors:\n")
            for error in errors:
                print(error)
            return

        current_json = generate_agent_card_json()
        print("✨ Agent Card Generated!\n")
        print(json.dumps(current_json, indent=2))


def on_download_click(b):
    global current_json
    errors = validate_card()
    if errors:
        with json_output:
            clear_output()
            print("⚠️ Cannot download - validation errors:\n")
            for error in errors:
                print(error)
        return

    current_json = generate_agent_card_json()

    safe_chars = " _-"
    agent_name_clean = "".join(
        c for c in agent_card_data["name"] if c.isalnum() or c in safe_chars
    )
    agent_name_clean = agent_name_clean.strip().replace(" ", "_").lower()
    filename = (
        f"agent_card_{agent_name_clean}.json" if agent_name_clean else "agent_card.json"
    )

    with open(filename, "w") as f:
        json.dump(current_json, f, indent=2)

    files.download(filename)
    print(f"✅ Downloaded: {filename}")


generate_button.on_click(on_generate_click)
download_button.on_click(on_download_click)

display(HTML("<h2 style='color: #a855f7;'>🟪 Generate Agent Card</h2>"))
display(widgets.HBox([generate_button, download_button]))
display(json_output)

---

## 📚 Next Steps

1. **Deploy your Agent Card** to `.well-known/agent-card.json` on your domain
2. **Test with A2A clients** using the official SDKs
3. **Join the community** at [a2a-protocol.org](https://a2a-protocol.org)

---

**Created by Paola Di Maio | A2A Protocol v1.0**